# Business Metrics — Python Reference

**Companion to:** `ds1_problem_framing.ipynb` — metric definitions, roles, and business context

**Purpose:** pandas recipes to compute every metric from the ds1 notes. Sample data is generated in §0 so every cell runs standalone. One section per metric, each ending with a visualization.

---

## §0 — Imports & Sample Data

Run this section first. All DataFrames defined here are reused throughout the notebook.

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date, timedelta

np.random.seed(42)

# ── date range ────────────────────────────────────────────────────────────────
START = pd.Timestamp('2024-01-01')
END   = pd.Timestamp('2024-03-31')
DATES = pd.date_range(START, END, freq='D')

# ── users ─────────────────────────────────────────────────────────────────────
N_USERS = 2000
channels = ['paid_search', 'organic', 'referral', 'social']
users = pd.DataFrame({
    'user_id':    range(1, N_USERS + 1),
    'created_at': pd.to_datetime(
        np.random.choice(pd.date_range('2023-10-01', '2024-03-01'), N_USERS)
    ),
    'channel': np.random.choice(channels, N_USERS, p=[0.35, 0.30, 0.20, 0.15]),
})

# ── events ────────────────────────────────────────────────────────────────────
N_EVENTS = 30_000
event_types = ['session', 'page_view', 'add_to_cart', 'checkout', 'purchase',
               'invite_sent', 'invite_accepted']
events = pd.DataFrame({
    'user_id':    np.random.choice(users['user_id'], N_EVENTS),
    'event_date': pd.to_datetime(np.random.choice(DATES, N_EVENTS)),
    'event_type': np.random.choice(
        event_types, N_EVENTS,
        p=[0.30, 0.25, 0.15, 0.12, 0.10, 0.05, 0.03]
    ),
})

# ── orders ────────────────────────────────────────────────────────────────────
purchase_events = events[events['event_type'] == 'purchase'].copy()
orders = purchase_events[['user_id', 'event_date']].rename(
    columns={'event_date': 'order_date'}
).copy()
orders.insert(0, 'order_id', range(1, len(orders) + 1))
orders['order_value'] = np.round(np.random.lognormal(mean=3.5, sigma=0.8, size=len(orders)), 2)

# ── marketing_spend ───────────────────────────────────────────────────────────
spend_rows = []
for d in pd.date_range('2024-01-01', '2024-03-31', freq='D'):
    for ch in channels:
        base = {'paid_search': 500, 'organic': 50, 'referral': 150, 'social': 200}[ch]
        spend_rows.append({'spend_date': d, 'channel': ch,
                           'spend': round(base * np.random.uniform(0.8, 1.2), 2)})
marketing_spend = pd.DataFrame(spend_rows)

# ── subscriptions (SaaS) ──────────────────────────────────────────────────────
sub_users = users.sample(600, random_state=42)['user_id'].tolist()
sub_rows = []
for uid in sub_users:
    fee = np.random.choice([9.99, 19.99, 49.99], p=[0.5, 0.35, 0.15])
    start_m = pd.Timestamp('2024-01-01')
    for m in pd.date_range(start_m, '2024-03-01', freq='MS'):
        churned = np.random.random() < 0.05
        sub_rows.append({'user_id': uid, 'month': m, 'monthly_fee': fee,
                         'status': 'churned' if churned else 'active'})
subscriptions = pd.DataFrame(sub_rows)

# ── ad_events ─────────────────────────────────────────────────────────────────
N_ADS = 20_000
ad_ids = [f'ad_{i:03d}' for i in range(1, 11)]
ad_events = pd.DataFrame({
    'user_id':    np.random.choice(users['user_id'], N_ADS),
    'event_date': pd.to_datetime(np.random.choice(DATES, N_ADS)),
    'ad_id':      np.random.choice(ad_ids, N_ADS),
    'event_type': np.random.choice(['impression', 'click'], N_ADS, p=[0.92, 0.08]),
})

# ── transactions (marketplace) ────────────────────────────────────────────────
N_TXN = 5_000
transactions = pd.DataFrame({
    'transaction_id':    range(1, N_TXN + 1),
    'buyer_id':          np.random.choice(users['user_id'], N_TXN),
    'seller_id':         np.random.choice(users['user_id'], N_TXN),
    'transaction_date':  pd.to_datetime(np.random.choice(DATES, N_TXN)),
    'transaction_value': np.round(np.random.lognormal(4.0, 0.9, N_TXN), 2),
})

# ── nps_responses ─────────────────────────────────────────────────────────────
nps_users = users.sample(800, random_state=7)['user_id'].tolist()
nps_responses = pd.DataFrame({
    'user_id':       nps_users,
    'response_date': pd.to_datetime(np.random.choice(DATES, len(nps_users))),
    'score':         np.random.choice(range(11), len(nps_users),
                                      p=[0.03,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.15,0.20,0.25]),
})

print("Sample data ready:")
print(f"  users:           {len(users):,} rows")
print(f"  events:          {len(events):,} rows")
print(f"  orders:          {len(orders):,} rows")
print(f"  marketing_spend: {len(marketing_spend):,} rows")
print(f"  subscriptions:   {len(subscriptions):,} rows")
print(f"  ad_events:       {len(ad_events):,} rows")
print(f"  transactions:    {len(transactions):,} rows")
print(f"  nps_responses:   {len(nps_responses):,} rows")
```

---

## §1 — DAU / WAU / MAU

**Formula:**
- DAU = unique users active on a given day
- WAU = unique users active in a given ISO week
- MAU = unique users active in a given calendar month

```python
# DAU
dau = (events.groupby('event_date')['user_id']
       .nunique()
       .reset_index(name='dau'))

# WAU
events['year_week'] = events['event_date'].dt.to_period('W')
wau = (events.groupby('year_week')['user_id']
       .nunique()
       .reset_index(name='wau'))
wau['week_start'] = wau['year_week'].dt.start_time

# MAU
events['year_month'] = events['event_date'].dt.to_period('M')
mau = (events.groupby('year_month')['user_id']
       .nunique()
       .reset_index(name='mau'))

print("DAU (first 5 days):")
print(dau.head())
print("\nMAU by month:")
print(mau)
```

```python
# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(dau['event_date'], dau['dau'], color='#3498DB', linewidth=1.5, alpha=0.9)
axes[0].set_title('Daily Active Users (DAU)', fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Unique users')
axes[0].grid(True, alpha=0.3); axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].bar(mau['year_month'].astype(str), mau['mau'], color='#2ECC71', edgecolor='white', width=0.5)
axes[1].set_title('Monthly Active Users (MAU)', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Unique users')
axes[1].grid(True, alpha=0.3, axis='y'); axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## §2 — DAU/MAU Ratio

**Formula:** DAU/MAU ratio = DAU ÷ MAU for the same calendar month

**Interpretation:** 0.20 → users active ~6 out of 30 days per month. Industry benchmarks: >0.20 is reasonable, >0.50 is exceptional (WhatsApp-tier).

```python
# Add month column to dau
dau['year_month'] = dau['event_date'].dt.to_period('M')

# Merge DAU with MAU on year_month
dau_mau = dau.merge(mau, on='year_month', how='left')
dau_mau['dau_mau_ratio'] = dau_mau['dau'] / dau_mau['mau']

print(dau_mau[['event_date', 'dau', 'mau', 'dau_mau_ratio']].head(10))
print("\nMean DAU/MAU ratio by month:")
print(dau_mau.groupby('year_month')['dau_mau_ratio'].mean().round(4))
```

```python
# Plot: 7-day rolling average of DAU/MAU
dau_mau_sorted = dau_mau.sort_values('event_date')
dau_mau_sorted['rolling_ratio'] = dau_mau_sorted['dau_mau_ratio'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(dau_mau_sorted['event_date'], dau_mau_sorted['dau_mau_ratio'],
                alpha=0.2, color='#9B59B6')
ax.plot(dau_mau_sorted['event_date'], dau_mau_sorted['rolling_ratio'],
        color='#9B59B6', linewidth=2, label='7-day rolling avg')
ax.axhline(0.20, color='#E74C3C', linestyle='--', linewidth=1.2, label='0.20 benchmark')
ax.set_title('DAU/MAU Ratio over Time', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('DAU/MAU ratio')
ax.legend(); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
```

---

## §3 — D1 / D7 / D30 Retention

**Formula:** Dn retention = users active on day n after signup ÷ cohort size (users who signed up on day 0)

```python
# Cohort: signup date per user
cohort = users[['user_id', 'created_at']].copy()
cohort['cohort_month'] = cohort['created_at'].dt.to_period('M')

# All event dates
activity = events[['user_id', 'event_date']].drop_duplicates()

# Merge and compute days since signup
merged = cohort.merge(activity, on='user_id', how='left')
merged['day_number'] = (merged['event_date'] - merged['created_at']).dt.days

# Cohort sizes
cohort_sizes = cohort.groupby('cohort_month')['user_id'].nunique().rename('cohort_size')

# Retention for each day 0–30
retention_rows = []
for day in range(0, 31):
    active_on_day = (merged[merged['day_number'] == day]
                     .groupby('cohort_month')['user_id'].nunique()
                     .rename('retained'))
    day_df = cohort_sizes.to_frame().join(active_on_day).fillna(0)
    day_df['day_number'] = day
    day_df['retention_rate'] = day_df['retained'] / day_df['cohort_size']
    retention_rows.append(day_df.reset_index())

retention = pd.concat(retention_rows, ignore_index=True)

# Summary: D1, D7, D30 per cohort month
pivot = retention[retention['day_number'].isin([1, 7, 30])].pivot(
    index='cohort_month', columns='day_number', values='retention_rate'
)
pivot.columns = ['D1', 'D7', 'D30']
print(pivot.round(4))
```

```python
# Retention curve plot
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12']
for i, (month, grp) in enumerate(retention.groupby('cohort_month')):
    grp_sorted = grp.sort_values('day_number')
    ax.plot(grp_sorted['day_number'], grp_sorted['retention_rate'],
            label=str(month), color=colors[i % len(colors)], linewidth=2, alpha=0.85)

ax.set_title('Retention Curves by Cohort Month (D0–D30)', fontweight='bold')
ax.set_xlabel('Days since signup'); ax.set_ylabel('Retention rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.axvline(1,  color='grey', linestyle=':', alpha=0.6, linewidth=1)
ax.axvline(7,  color='grey', linestyle=':', alpha=0.6, linewidth=1)
ax.axvline(30, color='grey', linestyle=':', alpha=0.6, linewidth=1)
ax.text(1,  0.02, 'D1',  fontsize=8, color='grey', ha='center')
ax.text(7,  0.02, 'D7',  fontsize=8, color='grey', ha='center')
ax.text(30, 0.02, 'D30', fontsize=8, color='grey', ha='center')
ax.legend(title='Cohort month'); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
```

---

## §4 — K-factor

**Formula:** K-factor = (invites sent per user) × (invite conversion rate)
- Invites sent per user = total invite_sent events ÷ unique inviters
- Invite conversion rate = invite_accepted ÷ invite_sent

```python
inv = events[events['event_type'].isin(['invite_sent', 'invite_accepted'])]

total_sent     = (inv['event_type'] == 'invite_sent').sum()
total_accepted = (inv['event_type'] == 'invite_accepted').sum()
total_inviters = inv[inv['event_type'] == 'invite_sent']['user_id'].nunique()

invites_per_user      = total_sent / total_inviters if total_inviters > 0 else 0
invite_conv_rate      = total_accepted / total_sent if total_sent > 0 else 0
k_factor              = invites_per_user * invite_conv_rate

print(f"Total invites sent:       {total_sent:,}")
print(f"Total invites accepted:   {total_accepted:,}")
print(f"Unique inviters:          {total_inviters:,}")
print(f"Invites per user:         {invites_per_user:.3f}")
print(f"Invite conversion rate:   {invite_conv_rate:.3f}")
print(f"K-factor:                 {k_factor:.3f}")
print(f"Viral?                    {'Yes (>1)' if k_factor > 1 else 'No (<1)'}")
```

```python
# Monthly K-factor trend
monthly_inv = events[events['event_type'].isin(['invite_sent', 'invite_accepted'])].copy()
monthly_inv['year_month'] = monthly_inv['event_date'].dt.to_period('M')

def calc_k(grp):
    sent     = (grp['event_type'] == 'invite_sent').sum()
    accepted = (grp['event_type'] == 'invite_accepted').sum()
    inviters = grp[grp['event_type'] == 'invite_sent']['user_id'].nunique()
    ipu = sent / inviters if inviters > 0 else 0
    icr = accepted / sent if sent > 0 else 0
    return pd.Series({'invites_per_user': ipu, 'invite_conv_rate': icr, 'k_factor': ipu * icr})

k_monthly = monthly_inv.groupby('year_month').apply(calc_k).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(k_monthly['year_month'].astype(str), k_monthly['k_factor'],
       color='#E67E22', edgecolor='white', width=0.4)
ax.axhline(1.0, color='#E74C3C', linestyle='--', linewidth=1.5, label='K=1 (viral threshold)')
ax.set_title('K-factor by Month', fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('K-factor')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
```

---

## §5 — NPS (Net Promoter Score)

**Formula:** NPS = % Promoters (9–10) − % Detractors (0–6)
- Passives (7–8) are in the denominator but excluded from both sides.

```python
def classify_nps(score):
    if score >= 9: return 'promoter'
    if score <= 6: return 'detractor'
    return 'passive'

nps_responses['nps_group'] = nps_responses['score'].apply(classify_nps)

counts = nps_responses['nps_group'].value_counts()
total  = len(nps_responses)

pct_promoters  = counts.get('promoter',  0) / total * 100
pct_detractors = counts.get('detractor', 0) / total * 100
pct_passives   = counts.get('passive',   0) / total * 100
nps_score      = pct_promoters - pct_detractors

print(f"Total responses:  {total:,}")
print(f"Promoters:        {counts.get('promoter',0):,}  ({pct_promoters:.1f}%)")
print(f"Passives:         {counts.get('passive',0):,}   ({pct_passives:.1f}%)")
print(f"Detractors:       {counts.get('detractor',0):,} ({pct_detractors:.1f}%)")
print(f"\nNPS Score:        {nps_score:.1f}")
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Score distribution
axes[0].hist(nps_responses['score'], bins=11, range=(-0.5, 10.5),
             color='#3498DB', edgecolor='white', alpha=0.85)
axes[0].axvspan(-0.5, 6.5,  alpha=0.08, color='#E74C3C', label='Detractors (0–6)')
axes[0].axvspan(6.5,  8.5,  alpha=0.08, color='#F39C12', label='Passives (7–8)')
axes[0].axvspan(8.5,  10.5, alpha=0.08, color='#2ECC71', label='Promoters (9–10)')
axes[0].set_title('NPS Score Distribution', fontweight='bold')
axes[0].set_xlabel('Score'); axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Breakdown bar
labels = ['Promoters', 'Passives', 'Detractors']
values = [pct_promoters, pct_passives, pct_detractors]
colors_bar = ['#2ECC71', '#F39C12', '#E74C3C']
bars = axes[1].bar(labels, values, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title(f'NPS Breakdown  (NPS = {nps_score:.1f})', fontweight='bold')
axes[1].set_ylabel('% of respondents')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
```

---

## §6 — Conversion Rate

**Formula:** Conversion rate = unique converters ÷ unique visitors

```python
funnel_steps = ['session', 'page_view', 'add_to_cart', 'checkout', 'purchase']

funnel_counts = {
    step: events[events['event_type'] == step]['user_id'].nunique()
    for step in funnel_steps
}
funnel_df = pd.DataFrame({'step': funnel_steps, 'users': [funnel_counts[s] for s in funnel_steps]})
funnel_df['step_conv_rate'] = funnel_df['users'] / funnel_df['users'].shift(1)
funnel_df['overall_conv_rate'] = funnel_df['users'] / funnel_df['users'].iloc[0]

print(funnel_df.round(4).to_string(index=False))
print(f"\nTop-of-funnel → Purchase conversion: "
      f"{funnel_df.iloc[-1]['overall_conv_rate']:.2%}")
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Funnel chart
colors_f = ['#2980B9', '#3498DB', '#5DADE2', '#85C1E9', '#AED6F1']
bars = axes[0].barh(funnel_df['step'], funnel_df['users'],
                    color=colors_f, edgecolor='white')
for bar, val in zip(bars, funnel_df['users']):
    axes[0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Conversion Funnel — Unique Users', fontweight='bold')
axes[0].set_xlabel('Unique users')
axes[0].invert_yaxis()
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Monthly conversion rate
events_monthly = events.copy()
events_monthly['year_month'] = events_monthly['event_date'].dt.to_period('M')
monthly_conv = events_monthly.groupby('year_month').apply(
    lambda g: pd.Series({
        'visitors':   g[g['event_type'] == 'session']['user_id'].nunique(),
        'converters': g[g['event_type'] == 'purchase']['user_id'].nunique(),
    })
).reset_index()
monthly_conv['conversion_rate'] = monthly_conv['converters'] / monthly_conv['visitors'].replace(0, np.nan)

axes[1].plot(monthly_conv['year_month'].astype(str), monthly_conv['conversion_rate'] * 100,
             marker='o', color='#2ECC71', linewidth=2.5, markersize=8)
axes[1].set_title('Monthly Conversion Rate', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Conversion rate (%)')
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## §7 — LTV (Lifetime Value)

**Formula:** LTV = avg order value × purchase frequency × avg customer lifespan (years)

```python
# Per-customer aggregates
cust_orders = orders.groupby('user_id').agg(
    order_count   = ('order_id',    'count'),
    total_revenue = ('order_value', 'sum'),
    avg_order_val = ('order_value', 'mean'),
    first_order   = ('order_date',  'min'),
    last_order    = ('order_date',  'max'),
).reset_index()

cust_orders['lifespan_days']  = (cust_orders['last_order'] - cust_orders['first_order']).dt.days
cust_orders['lifespan_years'] = cust_orders['lifespan_days'] / 365.0

# Overall LTV
avg_order_val   = cust_orders['avg_order_val'].mean()
avg_frequency   = cust_orders['order_count'].mean()
avg_lifespan    = cust_orders['lifespan_years'].mean()
ltv             = avg_order_val * avg_frequency * avg_lifespan

print(f"Avg order value:          ${avg_order_val:.2f}")
print(f"Avg purchase frequency:   {avg_frequency:.2f} orders")
print(f"Avg customer lifespan:    {avg_lifespan:.3f} years ({avg_lifespan*365:.1f} days)")
print(f"LTV:                      ${ltv:.2f}")

# LTV by channel
cust_channel = cust_orders.merge(users[['user_id', 'channel']], on='user_id')
ltv_channel = cust_channel.groupby('channel').apply(
    lambda g: pd.Series({
        'customers':   len(g),
        'avg_order_val': g['avg_order_val'].mean(),
        'avg_frequency': g['order_count'].mean(),
        'avg_lifespan':  g['lifespan_years'].mean(),
    })
).reset_index()
ltv_channel['ltv'] = (ltv_channel['avg_order_val']
                      * ltv_channel['avg_frequency']
                      * ltv_channel['avg_lifespan'])
print("\nLTV by channel:")
print(ltv_channel[['channel','customers','avg_order_val','avg_frequency','ltv']].round(2)
      .sort_values('ltv', ascending=False).to_string(index=False))
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cust_orders['total_revenue'], bins=40, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[0].axvline(cust_orders['total_revenue'].mean(), color='#E74C3C', linestyle='--',
                linewidth=2, label=f"Mean = ${cust_orders['total_revenue'].mean():.0f}")
axes[0].set_title('Customer Lifetime Revenue Distribution', fontweight='bold')
axes[0].set_xlabel('Total revenue per customer'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

ltv_sorted = ltv_channel.sort_values('ltv', ascending=True)
axes[1].barh(ltv_sorted['channel'], ltv_sorted['ltv'], color='#8E44AD', edgecolor='white', alpha=0.85)
for i, (_, row) in enumerate(ltv_sorted.iterrows()):
    axes[1].text(row['ltv'] + 0.5, i, f"${row['ltv']:.2f}", va='center', fontsize=9)
axes[1].set_title('LTV by Acquisition Channel', fontweight='bold')
axes[1].set_xlabel('LTV ($)')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## §8 — CAC (Customer Acquisition Cost)

**Formula:** CAC = total acquisition spend ÷ new customers acquired (same period)

```python
# Monthly spend
monthly_spend = (marketing_spend.groupby(
    marketing_spend['spend_date'].dt.to_period('M'))['spend']
    .sum().reset_index().rename(columns={'spend_date': 'year_month', 'spend': 'total_spend'}))

# Monthly new customers
users['year_month'] = users['created_at'].dt.to_period('M')
monthly_new = users.groupby('year_month').size().reset_index(name='new_customers')

cac_monthly = monthly_spend.merge(monthly_new, on='year_month', how='inner')
cac_monthly['cac'] = cac_monthly['total_spend'] / cac_monthly['new_customers']

print(cac_monthly.round(2).to_string(index=False))

# CAC by channel
channel_spend = (marketing_spend.groupby('channel')['spend']
                 .sum().reset_index(name='total_spend'))
channel_new = (users[users['created_at'] >= pd.Timestamp('2024-01-01')]
               .groupby('channel').size().reset_index(name='new_customers'))
cac_channel = channel_spend.merge(channel_new, on='channel')
cac_channel['cac'] = cac_channel['total_spend'] / cac_channel['new_customers']
print("\nCAC by channel:")
print(cac_channel.sort_values('cac').round(2).to_string(index=False))
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(cac_monthly['year_month'].astype(str), cac_monthly['cac'],
             marker='o', color='#E74C3C', linewidth=2.5, markersize=8)
axes[0].set_title('Monthly CAC', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('CAC ($)')
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

cac_sorted = cac_channel.sort_values('cac', ascending=True)
bars = axes[1].barh(cac_sorted['channel'], cac_sorted['cac'],
                    color='#E74C3C', edgecolor='white', alpha=0.85)
for bar, val in zip(bars, cac_sorted['cac']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'${val:.0f}', va='center', fontsize=9)
axes[1].set_title('CAC by Acquisition Channel', fontweight='bold')
axes[1].set_xlabel('CAC ($)')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## §9 — LTV/CAC Ratio

**Formula:** LTV/CAC = LTV ÷ CAC per acquisition channel

**Benchmark:** ≥ 3× healthy, 1–3× break-even, < 1× unsustainable.

```python
ltv_cac = ltv_channel[['channel', 'ltv']].merge(cac_channel[['channel', 'cac']], on='channel')
ltv_cac['ltv_cac_ratio'] = ltv_cac['ltv'] / ltv_cac['cac']
ltv_cac['status'] = ltv_cac['ltv_cac_ratio'].apply(
    lambda x: 'Healthy (≥3×)' if x >= 3 else ('Break-even (1–3×)' if x >= 1 else 'Unsustainable (<1×)')
)
print(ltv_cac[['channel','ltv','cac','ltv_cac_ratio','status']].round(2).to_string(index=False))
```

```python
fig, ax = plt.subplots(figsize=(10, 4))
colors_ltv = ltv_cac['ltv_cac_ratio'].apply(
    lambda x: '#2ECC71' if x >= 3 else ('#F39C12' if x >= 1 else '#E74C3C')
)
bars = ax.bar(ltv_cac['channel'], ltv_cac['ltv_cac_ratio'],
              color=colors_ltv, edgecolor='white', width=0.5)
for bar, val in zip(bars, ltv_cac['ltv_cac_ratio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(3.0, color='#2ECC71', linestyle='--', linewidth=1.5, label='3× healthy threshold')
ax.axhline(1.0, color='#E74C3C', linestyle='--', linewidth=1.5, label='1× break-even')
ax.set_title('LTV/CAC Ratio by Acquisition Channel', fontweight='bold')
ax.set_ylabel('LTV/CAC ratio')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
```

---

## §10 — MRR & MRR Growth Rate

**Formula:**
- MRR = sum of monthly fees for all active subscribers in a month
- MRR growth rate = (MRR_t − MRR_{t−1}) ÷ MRR_{t−1}

```python
active_subs = subscriptions[subscriptions['status'] == 'active'].copy()
active_subs['year_month'] = active_subs['month'].dt.to_period('M')

mrr = (active_subs.groupby('year_month')['monthly_fee']
       .sum().reset_index(name='mrr'))
mrr['prev_mrr']     = mrr['mrr'].shift(1)
mrr['mrr_growth']   = (mrr['mrr'] - mrr['prev_mrr']) / mrr['prev_mrr']

print(mrr.round(2).to_string(index=False))

# MRR by fee tier
mrr_tier = (active_subs.groupby(['year_month', 'monthly_fee'])['user_id']
            .nunique().reset_index(name='subscribers'))
mrr_tier['mrr_contribution'] = mrr_tier['monthly_fee'] * mrr_tier['subscribers']
print("\nMRR by tier:")
print(mrr_tier.groupby('monthly_fee')['mrr_contribution'].sum().round(2))
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(mrr['year_month'].astype(str), mrr['mrr'],
            color='#2980B9', edgecolor='white', width=0.5)
for i, row in mrr.iterrows():
    axes[0].text(i, row['mrr'] + 100, f"${row['mrr']:,.0f}",
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Monthly Recurring Revenue (MRR)', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('MRR ($)')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

mrr_valid = mrr.dropna(subset=['mrr_growth'])
bar_colors = ['#2ECC71' if v >= 0 else '#E74C3C' for v in mrr_valid['mrr_growth']]
axes[1].bar(mrr_valid['year_month'].astype(str), mrr_valid['mrr_growth'] * 100,
            color=bar_colors, edgecolor='white', width=0.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('MRR Growth Rate (Month-over-Month)', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Growth rate (%)')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## §11 — CTR (Click-Through Rate)

**Formula:** CTR = clicks ÷ impressions

```python
# Daily CTR
ctr_daily = (ad_events.groupby(['event_date', 'event_type'])
             .size().unstack(fill_value=0).reset_index())
ctr_daily.columns.name = None
ctr_daily['ctr'] = ctr_daily['click'] / ctr_daily['impression'].replace(0, np.nan)

print("Daily CTR (first 5):")
print(ctr_daily.head())
print(f"\nOverall CTR: {ctr_daily['click'].sum() / ctr_daily['impression'].sum():.2%}")

# CTR by ad
ctr_ad = (ad_events.groupby(['ad_id', 'event_type'])
          .size().unstack(fill_value=0).reset_index())
ctr_ad.columns.name = None
ctr_ad['ctr'] = ctr_ad['click'] / ctr_ad['impression'].replace(0, np.nan)
print("\nCTR by ad:")
print(ctr_ad.sort_values('ctr', ascending=False).round(4).to_string(index=False))
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ctr_sorted = ctr_daily.sort_values('event_date')
ctr_sorted['rolling_ctr'] = ctr_sorted['ctr'].rolling(7).mean()
axes[0].plot(ctr_sorted['event_date'], ctr_sorted['rolling_ctr'] * 100,
             color='#F39C12', linewidth=2)
axes[0].fill_between(ctr_sorted['event_date'], ctr_sorted['rolling_ctr'] * 100,
                     alpha=0.15, color='#F39C12')
axes[0].set_title('Daily CTR (7-day rolling avg)', fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('CTR (%)')
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

ctr_ad_sorted = ctr_ad.sort_values('ctr', ascending=True)
axes[1].barh(ctr_ad_sorted['ad_id'], ctr_ad_sorted['ctr'] * 100,
             color='#F39C12', edgecolor='white', alpha=0.85)
axes[1].set_title('CTR by Ad', fontweight='bold')
axes[1].set_xlabel('CTR (%)')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()
```

---

## §12 — GMV (Gross Merchandise Value)

**Formula:** GMV = sum of all completed transaction values in the period

```python
transactions['year_month'] = transactions['transaction_date'].dt.to_period('M')

gmv_monthly = transactions.groupby('year_month').agg(
    total_transactions = ('transaction_id',    'count'),
    unique_buyers      = ('buyer_id',          'nunique'),
    gmv                = ('transaction_value', 'sum'),
    avg_txn_value      = ('transaction_value', 'mean'),
).reset_index()

print(gmv_monthly.round(2).to_string(index=False))
print(f"\nTotal GMV (Q1 2024): ${gmv_monthly['gmv'].sum():,.2f}")
print(f"Avg transaction:     ${gmv_monthly['avg_txn_value'].mean():,.2f}")
```

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(gmv_monthly['year_month'].astype(str), gmv_monthly['gmv'],
            color='#1ABC9C', edgecolor='white', width=0.5)
for i, row in gmv_monthly.iterrows():
    axes[0].text(i, row['gmv'] + 500, f"${row['gmv']:,.0f}",
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Monthly GMV', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('GMV ($)')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].hist(transactions['transaction_value'], bins=50,
             color='#1ABC9C', edgecolor='white', alpha=0.85)
axes[1].axvline(transactions['transaction_value'].median(), color='#E74C3C', linestyle='--',
                linewidth=2, label=f"Median = ${transactions['transaction_value'].median():.0f}")
axes[1].set_title('Transaction Value Distribution', fontweight='bold')
axes[1].set_xlabel('Transaction value ($)'); axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
```

---

## Quick Reference — pandas Cheat Sheet

| Metric | Key pandas operation |
|---|---|
| DAU/WAU/MAU | `groupby('period')['user_id'].nunique()` |
| DAU/MAU ratio | merge DAU and MAU on period, divide |
| D1/D7/D30 retention | merge cohort + activity, `(event_date - created_at).dt.days` |
| K-factor | `(sent / inviters) * (accepted / sent)` |
| NPS | `% promoters - % detractors` after binning scores |
| Conversion rate | `nunique()` per funnel step, divide |
| LTV | `mean(avg_order_val) * mean(order_count) * mean(lifespan_years)` |
| CAC | `total_spend / new_customers` after merging spend + users |
| LTV/CAC | merge LTV and CAC tables, divide |
| MRR | `groupby('month')['monthly_fee'].sum()` on active subs |
| MRR growth rate | `.shift(1)` and `(mrr - prev) / prev` |
| CTR | `clicks / impressions` after `groupby + size().unstack()` |
| GMV | `groupby('month')['transaction_value'].sum()` |
